## ReadMe

This notebook is used to prepare global raster files to a specific country, covering both population and hazards. It takes as input global rasters and exports these same raster but cropped to a single country. This is done to speed-up processing time when computing population and location exposure.

Users only need to indicate country 3-letters iso_code (e.g., 'MOZ' or 'MDG') and the hazard list. All hazard layers are covered by defalut.

This notebook should be executed before data processing (data_processing_regions.ipynb / data_processing_schools.ipynb)

All file paths and custom parameters are defined on file constants.py

## Custom Functions

In [23]:
def create_folder_structure(iso_code):
    """
    This function creates the basic folder structure for a new country.
    """
    newpath = PATH + iso_code.lower()
    if not os.path.exists(newpath):
        os.makedirs(newpath)
    
    for subfolder in FOLDER_STRUCTURE_LIST:
        newpath = PATH + iso_code.lower() + '/' + subfolder
        if not os.path.exists(newpath):
            os.makedirs(newpath)

    return

In [24]:
def prepare_population_data(iso_code):
    """
    This function crops the global WorldPop population data to fit the country (iso_code)
    boundaries and export the resulting file as a tif file.
    """
    #Parameters GADM_ADMIN_VECTOR_PATH and POPULATION_RASTER_PATH are
    #declared on file constants.py
    input_admin_df = gpd.read_file(GADM_ADMIN_VECTOR_PATH['admin0'])
    country_admin_df = input_admin_df[input_admin_df['GID_0'] == iso_code]
    COUNTRY_POPULATION_RASTER_PATH = POPULATION_RASTER_PATH.replace('wrl',iso_code.lower())
    #Funcion clip_raster is declared on file utils.py. It doesn't return any value, only
    #export the resulting tif to the COUNTRY_POPULATION_RASTER_PATH location
    clip_raster(country_admin_df,POPULATION_RASTER_PATH,COUNTRY_POPULATION_RASTER_PATH)

    return
    

In [25]:
def prepare_infrastructure_data(iso_code, infrastructure_layer):
    """
    This function crops the global infrastructure layers to fit the country (iso_code)
    boundaries and export the resulting file as a geojson file.
    """
    #Parameters GADM_ADMIN_VECTOR_PATH and GLOBAL_INFRASTRUCTURE_LOCATION_PATH are
    #declared on file constants.py
    input_admin_df = gpd.read_file(GADM_ADMIN_VECTOR_PATH['admin0'])
    country_admin_df = input_admin_df[input_admin_df['GID_0'] == iso_code]
    INFRASTRUCTURE_LOCATION_PATH = GLOBAL_INFRASTRUCTURE_LOCATION_PATH[infrastructure_layer]
    COUNTRY_INFRASTRUCTURE_LOCATION_PATH = INFRASTRUCTURE_LOCATION_PATH.replace('wrl',iso_code.lower())
    #Funcion clip_raster is declared on file utils.py. It doesn't return any value, only
    #export the resulting tif to the COUNTRY_POPULATION_RASTER_PATH location
    clip_vector(country_admin_df,INFRASTRUCTURE_LOCATION_PATH,COUNTRY_INFRASTRUCTURE_LOCATION_PATH)

    return
    

In [26]:
def prepare_hazard_data(iso_code, hazard):
    """
    This function crops the global hazard raster data to fit the country (iso_code)
    boundaries and export the resulting file as a tif file. For flood and cyclone the
    operation is repeated for each returning period.
    """

    #Parameters GADM_ADMIN_VECTOR_PATH, INPUT_RASTER_PATH and HAZARD_RETURN_PERIOD are
    #declared on file constants.py
    input_admin_df = gpd.read_file(GADM_ADMIN_VECTOR_PATH['admin0'])
    country_admin_df = input_admin_df[input_admin_df['GID_0'] == iso_code]

    if hazard == 'heatwave_future':
        print('No raster for heatwave_future')
        return
    elif hazard == 'heatwave_current':
        return_period_list = ['']
    else:
        return_period_list = HAZARD_RETURN_PERIOD[hazard]
        
    for return_period in return_period_list:   
        INPUT_RASTER_PATH, COUNTRY_HAZARD_RASTER_PATH = prepare_raster_path(hazard, iso_code, return_period)
        #Funcion clip_raster is declared on file utils.py. It doesn't return any value, only
        #export the resulting tif to the COUNTRY_HAZARD_RASTER_PATH location
        clip_raster(country_admin_df, INPUT_RASTER_PATH,COUNTRY_HAZARD_RASTER_PATH)

    return
    

## Run Code

In [27]:
import geopandas as gpd
import os
from constants import *
from utils import *


In [32]:
########################## USER input ####################
#iso_code_list = ['VNM','NPL','UKR']
#hazard_list = ['cyclone_current','flood_current','cyclone_future','flood_future','earthquake']

iso_code_list = ['MOZ']
hazard_list = ['flood_current','earthquake']
infrastructure_list = ['health']
############################################################

In [33]:

#Execute code to crop both population, infrastructure and hazard layers for a given country and hazard list

for iso_code in iso_code_list:
    print(iso_code)
    create_folder_structure(iso_code)
    ##Prepare population raster
    prepare_population_data(iso_code)
    
    for infrastructure_layer in infrastructure_list:
        print(infrastructure_layer)
        ##Prepare infrastructure vector layer
        prepare_infrastructure_data(iso_code, infrastructure_layer)
    
    for hazard in hazard_list:
        print(hazard)
        ##Prepare hazard raster
        prepare_hazard_data(iso_code, hazard)

        
        

MOZ
health
flood_current
earthquake
